[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/14_memory.ipynb)

# 14. Memory

**Part 3: Context** · about 25 minutes

Everything so far has been about fitting one task into one context. But every run of our agent starts from zero. Run it on the same project ten times and it rediscovers the same things ten times: how to run the tests, where the code lives, which odd conventions the project has.

**Memory** is how an agent carries what it learned from one run to the next. The simplest version that works is what most coding agents actually use: **a plain text file of notes** that gets loaded into the system prompt at the start of every run, and that the agent can add to.

You've probably seen these files. Many projects keep an `AGENTS.md`, and Claude Code reads a `CLAUDE.md`. They're memory: short, human-readable notes about the project that every run starts with.

**By the end you will have:**

- a `remember` tool and a memory file the agent maintains
- three runs on the same project: no memory, memory, and memory that's gone **stale**
- the rules that keep memory helpful instead of harmful

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import tempfile
from datetime import date
from pathlib import Path

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

## 1. A project with a quirk

The tests live in a `tests/` folder without an `__init__.py`. That's common, and it means the obvious command, `python -m unittest`, quietly finds **zero** tests. You need `python -m unittest discover -s tests`. It's exactly the kind of thing an agent has to work out once, and should never have to work out again.

In [3]:
workspace = Path(tempfile.mkdtemp(prefix="agent-workspace-")).resolve()

(workspace / "greet.py").write_text('def greet(name):\n    return f"Namaste, {name}!"\n')
(workspace / "tests").mkdir()
(workspace / "tests" / "test_greet.py").write_text('''\
import unittest

from greet import greet


class GreetTests(unittest.TestCase):
    def test_greet(self):
        self.assertEqual(greet("Asha"), "Namaste, Asha!")
''')

164

## 2. A memory file and a `remember` tool

Some rules, each one learned the hard way by people building agents:

- **Plain text in the project.** You can read it, edit it and review changes to it like any other file.
- **Short, durable facts only.** "Run tests with X" belongs there. "I'm halfway through the refactor" doesn't, because it's stale by tomorrow.
- **Update in place.** A note that starts with the same label as an existing one *replaces* it, so the file doesn't fill up with contradictions.
- **A size limit.** Memory is loaded on every call (remember notebook 11), so it has to earn its tokens.
- **Every write is visible.** You'll see why in section 6.

In [4]:
MEMORY_FILE = workspace / ".agent" / "memory.md"
MEMORY_LIMIT = 2000  # characters


def load_memory():
    return MEMORY_FILE.read_text() if MEMORY_FILE.exists() else ""


def remember(note):
    """Save a short, durable fact. A note with the same label as an existing one replaces it."""
    note = " ".join(note.split())
    if len(note) > 300:
        return "Error: keep notes under 300 characters. Save one short fact at a time."
    label = note.split(":")[0].strip().lower() if ":" in note else None

    lines = [line for line in load_memory().splitlines() if line.startswith("- ")]
    if label:
        lines = [line for line in lines if not line[2:].split(" (")[0].split(":")[0].strip().lower() == label]
    lines.append(f"- {note} (saved {date.today().isoformat()})")

    text = "\n".join(lines) + "\n"
    if len(text) > MEMORY_LIMIT:
        return "Error: memory is full. Ask the user which notes to drop."
    MEMORY_FILE.parent.mkdir(exist_ok=True)
    MEMORY_FILE.write_text(text)
    print(f"    MEMORY WRITE: {note}")
    return f"Saved. Memory now has {len(lines)} notes."


# Updating in place, tested without a model:
remember("Run tests with: python -m unittest")
remember("Code style: four spaces, no tabs")
remember("Run tests with: python -m unittest discover -s tests")
print()
print(load_memory())
assert load_memory().count("Run tests with") == 1
MEMORY_FILE.unlink()
print("Checkpoint passed.")

    MEMORY WRITE: Run tests with: python -m unittest
    MEMORY WRITE: Code style: four spaces, no tabs
    MEMORY WRITE: Run tests with: python -m unittest discover -s tests

- Code style: four spaces, no tabs (saved 2026-11-13)
- Run tests with: python -m unittest discover -s tests (saved 2026-11-13)

Checkpoint passed.


## 3. The agent

It gets two tools: a shell restricted to a few safe commands (the full permission system from notebook 08 would work just as well), and `remember`. The important line is in `build_system_prompt`: **memory goes into the system prompt**, so the model sees it before it does anything.

In [5]:
def run_command(command):
    allowed = command.startswith(("python -m unittest", "ls", "cat ")) or \
              (command.startswith("find ") and "-exec" not in command and "-delete" not in command)
    if not allowed or any(op in command for op in (";", "&", "|", ">", "$(", "`")):
        return "Denied: only unittest, ls, cat and find are allowed here."
    command = re.sub(r"(?<![\w./-])python3?(?=\s|$)", lambda _: shlex.quote(sys.executable), command)
    env = {k: os.environ[k] for k in ("PATH", "SYSTEMROOT") if k in os.environ}
    try:
        done = subprocess.run(command, shell=True, cwd=workspace, env=env, capture_output=True, text=True, timeout=30)
    except subprocess.TimeoutExpired:
        return "stopped: took longer than 30 seconds"
    output = (done.stdout + done.stderr).replace(str(workspace), ".")[-2000:]
    return f"exit code: {done.returncode}\n{output}"


TOOLS = [
    {"type": "function", "function": {
        "name": "run_command", "description": "Run a shell command in the project folder (unittest, ls, cat, find).",
        "parameters": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]}}},
    {"type": "function", "function": {
        "name": "remember",
        "description": ("Save a short fact about this project for future runs, as 'Label: fact'. "
                        "Only save things that will still be true next time, like how to run the tests."),
        "parameters": {"type": "object", "properties": {"note": {"type": "string"}}, "required": ["note"]}}},
]
REGISTRY = {"run_command": run_command, "remember": remember}


def build_system_prompt():
    prompt = ("You are a coding agent. Check whether this project's tests pass. Note that a run that "
              "finds 0 tests hasn't tested anything. When you learn something that will save time next "
              "run, save it with remember.")
    memory = load_memory()
    if memory:
        prompt += f"\n\nWhat you remember about this project from earlier runs:\n{memory}"
    return prompt


def run_agent(task="Do the tests pass?", max_steps=10):
    messages = [{"role": "system", "content": build_system_prompt()}, {"role": "user", "content": task}]
    for step in range(1, max_steps + 1):
        payload = {"model": llm.model, "messages": messages, "tools": TOOLS}
        response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
        response.raise_for_status()
        message = response.json()["choices"][0]["message"]
        messages.append(message)
        if not message.get("tool_calls"):
            return {"answer": message.get("content"), "steps": step}
        for call in message["tool_calls"]:
            name = call["function"]["name"]
            try:
                arguments = json.loads(call["function"]["arguments"] or "{}")
                result = REGISTRY[name](**arguments) if name in REGISTRY else f"Error: unknown tool {name!r}"
            except Exception as error:
                result = f"Error: {type(error).__name__}: {error}"
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
            if name == "run_command":
                summary = " ".join(result.split("\n")[0:1] + [l for l in result.splitlines() if "Ran " in l])
                print(f"[{step}] $ {arguments.get('command')}\n     {summary}")
    return {"answer": None, "steps": max_steps}

## 4. Run 1: no memory

In [6]:
first = run_agent()
print("\nAnswer:", first["answer"])
print("Model calls:", first["steps"])
print("\nMemory file now:\n" + (load_memory() or "(empty)"))

[1] $ python -m unittest
     exit code: 5 Ran 0 tests in 0.000s


[2] $ find . -name 'test_*.py'
     exit code: 0


[3] $ python -m unittest discover -s tests
     exit code: 0 Ran 1 test in 0.000s


    MEMORY WRITE: Run tests with: python -m unittest discover -s tests



Answer: The tests pass (1 tests). I ran them with `python -m unittest discover -s tests`.
Model calls: 5

Memory file now:
- Run tests with: python -m unittest discover -s tests (saved 2026-11-13)



The agent tried the obvious command, noticed that zero tests ran, went looking, found the right command, and saved it. That's three or four commands of discovery.

**What a real model did.** Gemma 4 E2B never called `remember`. It listed the folder, ran the test file directly, and stopped, so memory stayed empty and the second run took 5 model calls against the first run's 4. A tool the model never uses is just a longer prompt. Getting models to save memory reliably usually takes an explicit last step ("before you finish, save anything the next run should know"), or code that saves certain facts itself.

## 5. Run 2: with memory

A brand new run, with nothing shared except the memory file.

In [7]:
second = run_agent()
print("\nAnswer:", second["answer"])
print("Model calls:", second["steps"])

[1] $ python -m unittest discover -s tests
     exit code: 0 Ran 1 test in 0.000s



Answer: The tests pass (1 tests). I ran them with `python -m unittest discover -s tests`.
Model calls: 2


In [8]:
# Checkpoint: the second run uses what the first one learned.
if llm.offline:
    assert second["steps"] < first["steps"]
    assert "Ran 0 tests" not in second["answer"]
print(f"Model calls: {first['steps']} without memory, {second['steps']} with memory.")
print("Checkpoint passed.")

Model calls: 5 without memory, 2 with memory.
Checkpoint passed.


## 6. Run 3: when memory goes stale

Memory has a failure mode that's worse than having no memory: **being wrong with confidence.** Let's rename the test folder, the way projects change all the time, and run again.

In [9]:
(workspace / "tests").rename(workspace / "checks")
print("Memory still says:", load_memory().strip())

third = run_agent()
print("\nAnswer:", third["answer"])
print("\nMemory file now:\n" + load_memory())

Memory still says: - Run tests with: python -m unittest discover -s tests (saved 2026-11-13)


[1] $ python -m unittest discover -s tests
     exit code: 1


[2] $ find . -name 'test_*.py'
     exit code: 0


[3] $ python -m unittest discover -s checks
     exit code: 0 Ran 1 test in 0.000s


    MEMORY WRITE: Run tests with: python -m unittest discover -s checks



Answer: The tests pass (1 tests). I ran them with `python -m unittest discover -s checks`.

Memory file now:
- Run tests with: python -m unittest discover -s checks (saved 2026-11-13)



In [10]:
# Checkpoint: the agent noticed the memory was wrong and corrected it, instead of trusting it.
if llm.offline:
    assert "checks" in load_memory() and "-s tests" not in load_memory()
print("Checkpoint passed.")

Checkpoint passed.


This worked because of two choices made earlier. The system prompt said a run that finds 0 tests "hasn't tested anything", so a broken memory showed up as a failure instead of a false "all good". And `remember` replaces notes with the same label, so the correction *replaced* the old fact instead of sitting next to it, which would have left the next run with two contradictory instructions.

With a real model, look at whether it trusted the memory blindly, verified it, or ignored it. Models vary a lot here. That's why memory should be written as *hints that can be checked* ("Run tests with: ...") rather than as unchecked claims ("all tests pass").

## 7. The dangerous part: who writes the memory?

Notebook 08 showed that text in a file can carry instructions for the model. Memory makes that worse, because a successful injection can **write itself into memory** and then sit in the system prompt of *every future run*. One bad README today becomes a standing instruction forever.

That's why every memory write in this notebook prints a `MEMORY WRITE` line, and why the memory is a plain file in the project that shows up in `git diff`. The defences, roughly from cheapest to strongest:

1. Make every write visible, as here.
2. Keep memory under version control, so a human reviews changes like any other code change.
3. Require approval for memory writes, using the approver from notebook 08.
4. Only allow writes in specific formats, such as `Run tests with: <command>`, and validate them.

## Recap

- Memory is notes that load into the system prompt at the start of every run.
- Store short, durable, checkable facts. Don't store task progress.
- Update in place by label, and keep it small, because it's paid for on every call.
- Stale memory is worse than none. Write notes so the agent can verify them.
- Memory writes are a prompt-injection target, so make them visible and reviewable.

## Exercises

1. Add a second kind of note, "Gotcha: ...", and get the agent to save one about the `__init__.py` quirk. Does it help in run 2?
2. Make memory writes require approval with `INTERACTIVE` and the approver from notebook 08.
3. Add a timestamp check: notes older than 30 days are loaded with "(may be out of date, check before relying on it)". Does that change how your model treats them?

## Clean up

In [11]:
shutil.rmtree(workspace)
print("Removed the temporary workspace.")

Removed the temporary workspace.


**Next:** [15. Agent v3](15_agent_v3.ipynb). Search, memory and context management in one agent, on a real-sized project.